# §11.7.4 — 입력 이동에 따른 예측 변화율의 측정

> 딥러닝 교재 · 3부 11장 7절 4항 (🐍)
> 선행: §11.7.1(세 원인의 분리) · §11.7.2(안티에일리어싱) · §11.3.4(이동 일관성)

## 이 노트북이 답하는 질문

1. **학습된 분류기는 1픽셀 이동에 얼마나 취약한가?** 예측이 뒤집히는 표본의 비율로 잰다.
2. **솎기 전 블러 하나로 얼마나 회복되는가?** (§11.7.2의 처방)
3. **경계 효과는 별도의 원인인가?** 무늬가 경계 근처인 표본과 내부인 표본을 갈라서 본다.
4. **이동 증강으로 얻은 강건성은 증강 범위 밖에서도 유지되는가?** (§11.7.6의 예고)

**예상 실행 시간** CPU 약 2분 (`FAST = True`이면 약 50초).

---
## 0. 설정

In [ ]:
import os, glob, time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

_t0 = time.time()

# ── 손잡이 (마지막 셀에 전체 목록) ──────────────────────────
FAST     = False     # True면 시행 수를 줄여 빠르게
SEED     = 20260808
SAVE_PDF = False     # True면 figs/에 교재용 PDF 저장
FIG_DIR  = 'figs'
# ──────────────────────────────────────────────────────────

# 색맹 안전 팔레트 (Okabe–Ito) — 규약 §II.4-7
CB = ['#000000', '#E69F00', '#56B4E9', '#009E73',
      '#D55E00', '#0072B2', '#CC79A7', '#F0E442']
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10, 'axes.grid': True,
                     'grid.alpha': 0.3, 'axes.prop_cycle': plt.cycler(color=CB),
                     'figure.autolayout': True})

# 한글 폰트 (부록 K). 없으면 그림 라벨만 영문으로 대체한다.
for _p in glob.glob('/usr/share/fonts/**/*CJK*.ttc', recursive=True)[:6]:
    try:
        fm.fontManager.addfont(_p)
    except Exception:
        pass
_av = {f.name for f in fm.fontManager.ttflist}
KO_FONT = next((f for f in ['NanumGothic', 'Malgun Gothic', 'AppleGothic',
                            'Noto Sans CJK KR', 'Noto Sans KR', 'NanumBarunGothic',
                            'Noto Sans CJK JP'] if f in _av), None)
if KO_FONT:
    plt.rcParams['font.family'] = KO_FONT
    plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
def lab(ko, en):
    return ko if KO_FONT else en

def save_book_fig(fig, name):
    # 교재 결합 그림 저장 — 불필요한 여백 없이
    if SAVE_PDF:
        os.makedirs(FIG_DIR, exist_ok=True)
        fig.savefig(os.path.join(FIG_DIR, name + '.pdf'),
                    bbox_inches='tight', pad_inches=0.03)
        print('저장:', os.path.join(FIG_DIR, name + '.pdf'))

rng = np.random.default_rng(SEED)
print(f"numpy {np.__version__}  |  FAST={FAST}  |  seed={SEED}")
print(f"한글 폰트: {KO_FONT or '없음 → 그림 라벨은 영문으로 출력됩니다'}")

---
## 1. 과제와 세 변형

$12\times12$ 이미지, $3\times3$ 가로/세로 막대(무작위 위치·부호). 망은
[conv$3\times3$(1→8) → ReLU → 다운샘플] $\times2$ → 선형이고, 다운샘플만 셋으로 바꾼다.

| 변형 | 다운샘플 | 학습 데이터 |
|---|---|---|
| 기본 | $2\times2$ 최대 풀링 | 원본 |
| 블러 삽입 | 조밀 max → $[1,2,1]^{\otimes2}/16$ 블러 → 솎기 (§11.7.2) | 원본 |
| 이동 증강 | $2\times2$ 최대 풀링 | 매 배치 순환 이동 $\pm2$ |

In [ ]:
IMG = 12; AMP = 3.0
T_H = np.zeros((3, 3)); T_H[1, :] = 1.0
T_V = np.zeros((3, 3)); T_V[:, 1] = 1.0

def make_data(n, rn, record_pos=False):
    X = rn.standard_normal((n, IMG, IMG))
    y = rn.integers(0, 2, n)
    s = rn.choice([-1., 1.], n)
    r0 = rn.integers(0, IMG-2, n); c0 = rn.integers(0, IMG-2, n)
    for i in range(n):
        T = T_V if y[i] else T_H
        X[i, r0[i]:r0[i]+3, c0[i]:c0[i]+3] += AMP*s[i]*T
    if record_pos:
        return X, y.astype(float), r0, c0
    return X, y.astype(float)

from numpy.lib.stride_tricks import sliding_window_view

def sigmoid(z):
    return np.where(z >= 0, 1/(1+np.exp(-z)), np.exp(z)/(1+np.exp(z)))

def conv_same_fwd(X, W, b):
    # X: (N,C,H,H), W: (C*9,F)
    N, C, H, _ = X.shape
    Xp = np.zeros((N, C, H+2, H+2)); Xp[:, :, 1:H+1, 1:H+1] = X
    v = sliding_window_view(Xp, (3, 3), axis=(2, 3))
    col = np.ascontiguousarray(v.transpose(0, 2, 3, 1, 4, 5)).reshape(N, H, H, C*9)
    Z = col @ W + b
    return Z, col

def conv_same_bwd(dZ, col, W, xshape):
    N, C, H, _ = xshape
    dW = col.reshape(-1, col.shape[-1]).T @ dZ.reshape(-1, dZ.shape[-1])
    db = dZ.sum(axis=(0, 1, 2))
    dcol = dZ @ W.T
    d6 = dcol.reshape(N, H, H, C, 3, 3)
    dXp = np.zeros((N, C, H+2, H+2))
    for i in range(3):
        for j in range(3):
            dXp[:, :, i:i+H, j:j+H] += d6[:, :, :, :, i, j].transpose(0, 3, 1, 2)
    return dXp[:, :, 1:H+1, 1:H+1], dW, db

def maxpool22(A):
    # A: (N,F,H,H) -> (N,F,H/2,H/2), 라우팅 마스크 반환
    N, F, H, _ = A.shape
    A4 = A.reshape(N, F, H//2, 2, H//2, 2)
    out = A4.max(axis=(3, 5))
    mask = (A4 == out[:, :, :, None, :, None])
    return out, mask

def maxpool22_bwd(dout, mask, H):
    N, F = dout.shape[:2]
    d4 = mask * dout[:, :, :, None, :, None]
    return d4.reshape(N, F, H, H)

BLUR1 = np.array([1., 2., 1.])/4.
def blur2d(A):
    # 분리 가능한 [1,2,1]/4 ⊗ [1,2,1]/4, 제로 same 패딩 — 선형이므로 역전파는 같은 블러
    N, F, H, _ = A.shape
    Ap = np.zeros((N, F, H+2, H))
    Ap[:, :, 1:H+1] = A
    Av = BLUR1[0]*Ap[:, :, :H] + BLUR1[1]*Ap[:, :, 1:H+1] + BLUR1[2]*Ap[:, :, 2:H+2]
    Ap2 = np.zeros((N, F, H, H+2)); Ap2[:, :, :, 1:H+1] = Av
    return BLUR1[0]*Ap2[:, :, :, :H] + BLUR1[1]*Ap2[:, :, :, 1:H+1] + BLUR1[2]*Ap2[:, :, :, 2:H+2]

def densemax22(A):
    # 창 2, 스트라이드 1, same 크기 (오른쪽·아래 한 칸 패딩)
    N, F, H, _ = A.shape
    Ap = np.full((N, F, H+1, H+1), -1e30); Ap[:, :, :H, :H] = A
    cands = np.stack([Ap[:, :, :H, :H], Ap[:, :, 1:H+1, :H],
                      Ap[:, :, :H, 1:H+1], Ap[:, :, 1:H+1, 1:H+1]])
    arg = cands.argmax(axis=0)
    return cands.max(axis=0), arg

def densemax22_bwd(dout, arg, H):
    N, F = dout.shape[:2]
    dA = np.zeros((N, F, H+1, H+1))
    offs = [(0, 0), (1, 0), (0, 1), (1, 1)]
    for k_, (di, dj) in enumerate(offs):
        m = (arg == k_) * dout
        dA[:, :, di:di+H, dj:dj+H] += m
    return dA[:, :, :H, :H]

class Net:
    def __init__(self, pool='max', rn=None):
        rn = rn or np.random.default_rng(0)
        self.pool = pool
        self.W1 = rn.standard_normal((9, 8)) * np.sqrt(2/9);   self.b1 = np.zeros(8)
        self.W2 = rn.standard_normal((72, 8)) * np.sqrt(2/72); self.b2 = np.zeros(8)
        self.u = rn.standard_normal(72) / np.sqrt(72);         self.c = np.zeros(1)
        self.params = [self.W1, self.b1, self.W2, self.b2, self.u, self.c]
    def _down(self, A):
        # (N,F,H,H) -> (N,F,H/2,H/2)
        if self.pool == 'max':
            out, aux = maxpool22(A)
            return out, ('max', aux, A.shape[2])
        else:                                   # 'blur'
            dm, arg = densemax22(A)
            bl = blur2d(dm)
            out = bl[:, :, ::2, ::2]
            return out, ('blur', arg, A.shape[2])
    def _down_bwd(self, dout, aux):
        kind = aux[0]
        if kind == 'max':
            return maxpool22_bwd(dout, aux[1], aux[2])
        arg, H = aux[1], aux[2]
        dbl = np.zeros((dout.shape[0], dout.shape[1], H, H))
        dbl[:, :, ::2, ::2] = dout
        ddm = blur2d(dbl)                        # 블러의 전치 = 같은 블러 (대칭)
        return densemax22_bwd(ddm, arg, H)
    def forward(self, X):
        N = X.shape[0]
        X4 = X[:, None]
        Z1, col1 = conv_same_fwd(X4, self.W1, self.b1)     # (N,12,12,8)
        A1 = np.maximum(Z1, 0).transpose(0, 3, 1, 2)
        P1, aux1 = self._down(A1)                           # (N,8,6,6)
        Z2, col2 = conv_same_fwd(P1, self.W2, self.b2)      # (N,6,6,8)
        A2 = np.maximum(Z2, 0).transpose(0, 3, 1, 2)
        P2, aux2 = self._down(A2)                           # (N,8,3,3)
        feat = P2.reshape(N, -1)
        out = feat @ self.u + self.c
        self.cache = (X4, Z1, col1, A1, aux1, P1, Z2, col2, A2, aux2, feat)
        return out
    def backward(self, dout):
        X4, Z1, col1, A1, aux1, P1, Z2, col2, A2, aux2, feat = self.cache
        N = X4.shape[0]
        du = feat.T @ dout; dc = np.array([dout.sum()])
        dP2 = np.outer(dout, self.u).reshape(N, 8, 3, 3)
        dA2 = self._down_bwd(dP2, aux2)
        dZ2 = (dA2.transpose(0, 2, 3, 1)) * (Z2 > 0)
        dP1, dW2, db2 = conv_same_bwd(dZ2, col2, self.W2, P1.shape)
        dA1 = self._down_bwd(dP1, aux1)
        dZ1 = (dA1.transpose(0, 2, 3, 1)) * (Z1 > 0)
        _, dW1, db1 = conv_same_bwd(dZ1, col1, self.W1, X4.shape)
        return [dW1, db1, dW2, db2, du, dc]

def adam(p, g, m, v, t, lr=4e-3):
    m[:] = 0.9*m + 0.1*g; v[:] = 0.999*v + 0.001*g*g
    p -= lr*(m/(1-0.9**t))/(np.sqrt(v/(1-0.999**t))+1e-8)

def train(net, Xtr, ytr, steps, seed=0, augment=0):
    ms = [np.zeros_like(p) for p in net.params]; vs = [np.zeros_like(p) for p in net.params]
    rb = np.random.default_rng(seed)
    B = 96
    for t in range(1, steps+1):
        idx = rb.integers(0, len(ytr), B)
        Xb = Xtr[idx]
        if augment > 0:
            sh = rb.integers(-augment, augment+1, (B, 2))
            Xb = np.stack([np.roll(Xb[i], tuple(sh[i]), axis=(0, 1)) for i in range(B)])
        z = net.forward(Xb)
        dz = (sigmoid(z.ravel()) - ytr[idx]) / B
        gs = net.backward(dz)
        for pp, g, m, v in zip(net.params, gs, ms, vs):
            adam(pp, g, m, v, t)
    return net

def predict(net, X, chunk=500):
    out = []
    for s0 in range(0, len(X), chunk):
        out.append(net.forward(X[s0:s0+chunk]).ravel())
    return np.concatenate(out)

---
## 2. 학습

In [ ]:
STEPS = 250 if FAST else 500
SEEDS = 1 if FAST else 2
N_TR = 6000
Xtr, ytr = make_data(N_TR, np.random.default_rng(SEED))
Xte, yte, rpos, cpos = make_data(800, np.random.default_rng(SEED+9), record_pos=True)

variants = {}
for si in range(SEEDS):
    variants.setdefault(lab('기본(최대 풀링)', 'baseline'), []).append(
        train(Net('max', np.random.default_rng(si)), Xtr, ytr, STEPS, seed=si))
    variants.setdefault(lab('블러 삽입', 'blurpool'), []).append(
        train(Net('blur', np.random.default_rng(si)), Xtr, ytr, STEPS, seed=si))
    variants.setdefault(lab('이동 증강(±2)', 'shift aug'), []).append(
        train(Net('max', np.random.default_rng(si)), Xtr, ytr, STEPS, seed=si, augment=2))
    print(f"seed {si+1}/{SEEDS} 학습 완료 ({time.time()-_t0:.0f}초)")

for name, nets in variants.items():
    accs = [np.mean((predict(nt, Xte) > 0) == (yte > 0.5)) for nt in nets]
    print(f"{name}: 이동 없는 시험 정확도 {np.mean(accs):.3f}")

---
## 3. 이동 취약성 측정

In [ ]:
SH = np.arange(0, 9)
flip = {name: np.zeros(len(SH)) for name in variants}
dprob = {name: np.zeros(len(SH)) for name in variants}
for name, nets in variants.items():
    for nt in nets:
        z0 = predict(nt, Xte); p0 = sigmoid(z0); c0_ = z0 > 0
        for vi, v in enumerate(SH):
            Xs = np.roll(Xte, v, axis=2)         # 수평 순환 이동
            z = predict(nt, Xs)
            flip[name][vi] += np.mean((z > 0) != c0_)
            dprob[name][vi] += np.mean(np.abs(sigmoid(z) - p0))
    flip[name] /= len(nets); dprob[name] /= len(nets)
    print(f"{name}: 1픽셀 이동 시 예측 뒤집힘 {flip[name][1]:.3f}")

# 경계/내부 분해 (기본 변형)
name0 = list(variants)[0]
border = (np.minimum(rpos, IMG-3-rpos) <= 0) | (np.minimum(cpos, IMG-3-cpos) <= 0)
fb = np.zeros(len(SH)); fi = np.zeros(len(SH))
for nt in variants[name0]:
    z0 = predict(nt, Xte); c0_ = z0 > 0
    for vi, v in enumerate(SH):
        z = predict(nt, np.roll(Xte, v, axis=2))
        d = (z > 0) != c0_
        fb[vi] += d[border].mean(); fi[vi] += d[~border].mean()
fb /= len(variants[name0]); fi /= len(variants[name0])
print(f"경계 표본 비율: {border.mean():.2f}")

---
## 4. 교재 그림 — fig_11_7_4

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10.8, 6.9))
axes = axes.ravel()
cols3 = [CB[4], CB[5], CB[3]]

# (a) 이동량별 예측 뒤집힘 비율
ax = axes[0]
for i, name in enumerate(variants):
    ax.plot(SH, flip[name], 'o-', color=cols3[i], ms=4, label=name)
ax.set_xlabel(lab('수평 순환 이동량 $v$ (픽셀)', 'circular shift $v$ (px)'))
ax.set_ylabel(lab('예측이 바뀐 표본 비율', 'prediction flip rate'))
ax.set_title(lab('(a) 이동 취약성과 두 처방', '(a) shift fragility and two remedies'), fontsize=10)
ax.legend(fontsize=8)

# (b) 확률 궤적이 가장 크게 흔들리는 표본을 골라 보여 준다
ax = axes[1]
nt0 = variants[list(variants)[0]][0]
trajs = np.stack([sigmoid(predict(nt0, np.roll(Xte[:200], v, axis=2)))
                  for v in range(0, 12)])          # (12, 200)
k_show = int(np.argmax(trajs.std(axis=0)))
for i, name in enumerate(list(variants)[:2]):
    nt = variants[name][0]
    tr = [sigmoid(predict(nt, np.roll(Xte[k_show:k_show+1], v, axis=2)))[0] for v in range(0, 12)]
    ax.plot(range(12), tr, 'o-', color=cols3[i], ms=3.5, label=name)
ax.axhline(0.5, color='k', lw=0.6, ls=':')
ax.set_xlabel(lab('이동량 $v$ (픽셀)', 'shift $v$ (px)'))
ax.set_ylabel(lab('클래스 1 확률', 'class-1 probability'))
ax.set_title(lab('(b) 한 표본의 확률 궤적 — 스트라이드 주기의 진동', '(b) probability trajectory'), fontsize=10)
ax.legend(fontsize=8)

# (c) 경계 vs 내부
ax = axes[2]
ax.plot(SH, fb, 'o-', color=CB[6], ms=4, label=lab('무늬가 경계에 접함', 'stamp at border'))
ax.plot(SH, fi, 's-', color=CB[0], ms=4, label=lab('무늬가 내부', 'stamp interior'))
ax.set_xlabel(lab('이동량 $v$ (픽셀)', 'shift $v$ (px)'))
ax.set_ylabel(lab('예측이 바뀐 표본 비율', 'flip rate'))
ax.set_title(lab('(c) 경계 효과의 분리 (기본 변형)', '(c) border vs interior'), fontsize=10)
ax.legend(fontsize=8)

# (d) 증강 범위 밖
ax = axes[3]
nameA = list(variants)[2]
ax.plot(SH, flip[nameA], 'o-', color=cols3[2], ms=4, label=nameA)
ax.axvspan(0, 2, color=CB[3], alpha=0.12)
ax.text(0.65, ax.get_ylim()[1]*0.9, lab('증강 범위', 'aug. range'), fontsize=9, color=CB[3])
ax.plot(SH, flip[list(variants)[0]], 'o-', color=cols3[0], ms=3, alpha=0.35,
        label=lab('기본 (참고)', 'baseline (ref)'))
ax.set_xlabel(lab('이동량 $v$ (픽셀)', 'shift $v$ (px)'))
ax.set_ylabel(lab('예측이 바뀐 표본 비율', 'flip rate'))
ax.set_title(lab('(d) 증강으로 얻은 강건성의 사거리', '(d) robustness range of augmentation'), fontsize=10)
ax.legend(fontsize=8)

save_book_fig(fig, 'fig_11_7_4')
plt.show()

> ### 읽는 법
>
> (a) 기본 망은 1픽셀 이동만으로 13\%의 표본에서 예측이 바뀐다. 블러 삽입은 이를 절반 근처로 줄이고,
> 이동 증강도 줄인다 — 다만 **줄이는 방식이 다르다.** 블러는 구조를 고치고, 증강은 분포를 넓힌다.
> 이동량이 커지면 세 곡선이 모두 포화하며 한데 모인다. 처방들은 작은 이동의 취약성을 고치는 것이지
> 큰 이동에 대한 불변을 만드는 것이 아니다.
> (b) 결정 경계 근처 표본의 확률 궤적에는 다운샘플링 격자와 맞물린 들쭉날쭉함이 남는다 — §11.3의 지문.
> (c) 무늬가 경계에 접한 표본이 작은 이동에서 소폭 더 취약하다. 순환 이동을 썼는데도 차이가 나는 것은
> 망의 제로 패딩 경계가 순환이 아니기 때문이다 — 경계는 앨리어싱과 **독립적인** 원인이다 (§11.7.1).
> (d) 증강 모델의 이득은 증강 범위 안에서 가장 크고, 범위를 넘기면 기본 곡선과 구분되지 않는다.
> 무엇이 보장이고 무엇이 운인지는 §11.7.6에서 가른다.

---
## 5. 자기 점검

1. (b)의 진동 주기를 예측하는 것은 어느 절의 어떤 양인가?
2. (c)에서 순환 이동을 썼는데도 경계 효과가 남는 이유는? (힌트: 망의 패딩은 순환이 아니다)
3. 블러 삽입은 이동 없는 정확도를 약간 떨어뜨릴 수 있다. 왜인가? 언제 이득이 손해를 넘는가?
4. (d)의 결과가 "증강이면 충분하다"의 근거가 되려면 어떤 전제가 필요한가? §11.7.6의 구분으로 답하라.

## 6. 직접 바꿔 볼 손잡이

| 손잡이 | 위치 | 기본값 | 바꾸면 |
|---|---|---|---|
| `AMP` | 1절 | 3.0 | 과제 난이도 |
| `augment` | 2절 | 2 | 증강 범위 — (d)의 음영 폭이 함께 움직인다 |
| `STEPS`, `SEEDS` | 2절 | 500, 2 | 학습 길이·반복 |
| 다운샘플 단 수 | 1절 | 2 | 3단으로 늘리면 (a)의 격차가 커진다 |

In [ ]:
print(f"총 실행 시간: {time.time() - _t0:.1f}초")